# Hybrid RAG Q&A Assistant
### Advanced Machine Learning Project
### Ahmed Hatem, Ahmed Khattab, Yassin Bedier


### Package Installation & Imports

In [1]:
# Install required packages
!pip install rank_bm25 sentence-transformers faiss-cpu datasets gradio plotly openai nltk pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.7 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import math
import pandas as pd
import numpy as np
import plotly.express as px

# --- Configuration ---
OPENROUTER_API_KEY = 'USE_API_KEY_HERE'
MODEL = 'openai/gpt-oss-120b:free'
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
CHUNK_SIZE = 400

### Dataset Loading & Preprocessing

In [3]:
import re

def clean(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

JSON_FILES = [
    'DataSet/top_ai_questions.json',
    'DataSet/top_datascience_questions.json'
]

records = []
for fpath in JSON_FILES:
    with open(fpath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for q in data:
        ans_text = clean(q.get('body', ''))
        if q.get('answers'):
            best = max(q['answers'], key=lambda a: a.get('score', 0))
            ans_text = clean(best.get('body', ''))

        if len(ans_text) < 10: continue

        records.append({
            'question_id': q.get('question_id', 0),
            'title': clean(q.get('title', '')),
            'tags': ', '.join(q.get('tags', [])),
            'answer': ans_text,
        })

df = pd.DataFrame(records).drop_duplicates('question_id').reset_index(drop=True)
print(f"Loaded {len(df)} Q&A pairs.")

# Chunking
corpus = []
for idx, row in df.iterrows():
    text = row['answer']
    for i in range(0, len(text), CHUNK_SIZE):
        corpus.append({
            'doc_id': idx,
            'passage': text[i:i+CHUNK_SIZE],
            'title': row['title'],
            'tags': row['tags']
        })
print(f"Total passages after chunking: {len(corpus)}")

Loaded 1485 Q&A pairs.
Total passages after chunking: 3629


### Milestone 1: BM25 Retrieval

In [5]:
import nltk

nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [12]:
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

def tokenise(text):
    tokens = word_tokenize(text.lower())
    return [t for t in tokens if t.isalnum() and t not in STOPWORDS]

tokenised_corpus = [tokenise(p['passage']) for p in corpus]
bm25 = BM25Okapi(tokenised_corpus)

def bm25_retrieve(query, top_k=10):
    q_tokens = tokenise(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idx):
        results.append({
            'rank': rank + 1,
            'passage': corpus[idx]['passage'],
            'title': corpus[idx]['title'],
            'tags': corpus[idx]['tags'],
            'doc_id': corpus[idx]['doc_id'],
            'score': float(scores[idx]),
            'method': 'BM25'
        })
    return results

sample_queries = [
    'What is the difference between artificial intelligence and machine learning?',
    'Why do transformers outperform RNNs?',
    'What is self-supervised learning?'
]

print('--- BM25 Sample Queries ---')
for q in sample_queries:
    print(f'\nQuery: {q}')
    hits = bm25_retrieve(q)
    for h in hits[:2]:
        print(f"  [{h['rank']}] {h['title'][:70]}")

--- BM25 Sample Queries ---

Query: What is the difference between artificial intelligence and machine learning?
  [1] Difference between machine learning and artificial intelligence
  [2] What are the top artificial intelligence journals?

Query: Why do transformers outperform RNNs?
  [1] Why does the transformer do better than RNN and LSTM in long-range con
  [2] Why is the decoder not a part of BERT architecture?

Query: What is self-supervised learning?
  [1] What is self-supervised learning in machine learning?
  [2] What is the relationship between these two taxonomies for machine lear


### Milestone 3: Semantic Search with MiniLM

In [7]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
passages_text = [p['passage'] for p in corpus]

print('Encoding corpus...')
corpus_embeddings = embedder.encode(passages_text, convert_to_numpy=True, normalize_embeddings=True)

dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(corpus_embeddings)

def semantic_retrieve(query, top_k=10):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = faiss_index.search(q_emb, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
        results.append({
            'rank': rank + 1,
            'passage': corpus[idx]['passage'],
            'title': corpus[idx]['title'],
            'tags': corpus[idx]['tags'],
            'doc_id': corpus[idx]['doc_id'],
            'score': float(score),
            'method': 'Semantic'
        })
    return results

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding corpus...


### Milestone 2: Evaluation (MAP, MRR, nDCG)

In [13]:
EVAL_QS = [
    {'query': 'What is the difference between AI and machine learning?', 'keywords': ['artificial', 'intelligence', 'machine', 'learning', 'ai']},
    {'query': 'Why do transformers outperform RNNs?', 'keywords': ['transformer', 'rnn', 'attention']},
    {'query': 'What is self-supervised learning?', 'keywords': ['self-supervised', 'unsupervised', 'labels']},
    {'query': 'What is model-free reinforcement learning?', 'keywords': ['model-free', 'reinforcement', 'rl']},
    {'query': 'How to prevent overfitting in neural networks?', 'keywords': ['overfitting', 'dropout', 'regularization']},
]

def is_relevant(hit, keywords):
    text = (hit['title'] + ' ' + hit['passage']).lower()
    return any(kw in text for kw in keywords)

def evaluate(retrieve_fn, label):
    ap_list, rr_list, ndcg_list = [], [], []
    for eq in EVAL_QS:
        hits = retrieve_fn(eq['query'])

        # MAP
        n_rel = sum(1 for h in hits if is_relevant(h, eq['keywords']))
        ap = sum((sum(1 for h in hits[:i+1] if is_relevant(h, eq['keywords'])) / (i+1))
                 for i, h in enumerate(hits) if is_relevant(h, eq['keywords'])) / max(n_rel, 1)
        ap_list.append(ap)

        # MRR
        rr = next((1.0/(i+1) for i, h in enumerate(hits) if is_relevant(h, eq['keywords'])), 0.0)
        rr_list.append(rr)

        # nDCG
        dcg = sum(1.0 / math.log2(i + 2) for i, h in enumerate(hits) if is_relevant(h, eq['keywords']))
        idcg = sum(1.0 / math.log2(i + 2) for i in range(min(10, n_rel)))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0)

    return {
        'Method': label,
        'MAP@10': round(np.mean(ap_list), 4),
        'MRR@10': round(np.mean(rr_list), 4),
        'nDCG@10': round(np.mean(ndcg_list), 4)
    }

res_bm25 = evaluate(bm25_retrieve, 'BM25')
res_sem = evaluate(semantic_retrieve, 'Semantic')

eval_df = pd.DataFrame([res_bm25, res_sem])
print(eval_df)

     Method  MAP@10  MRR@10  nDCG@10
0      BM25  0.9292     1.0   0.9748
1  Semantic  1.0000     1.0   1.0000


### Milestone 4: Hybrid RRF Fusion

In [14]:
def rrf_fuse(bm25_hits, sem_hits, k=60, top_n=5):
    scores = {}
    meta = {}

    for rank, hit in enumerate(bm25_hits):
        did = hit['doc_id']
        scores[did] = scores.get(did, 0.0) + 1.0 / (k + rank + 1)
        meta[did] = hit

    for rank, hit in enumerate(sem_hits):
        did = hit['doc_id']
        scores[did] = scores.get(did, 0.0) + 1.0 / (k + rank + 1)
        if did not in meta: meta[did] = hit

    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    results = []
    for rank, (did, score) in enumerate(sorted_docs[:top_n]):
        entry = dict(meta[did])
        entry.update({'rank': rank+1, 'score': round(score, 5), 'method': 'Hybrid'})
        results.append(entry)
    return results

def hybrid_retrieve(query):
    bm25_hits = bm25_retrieve(query)
    sem_hits = semantic_retrieve(query)
    return rrf_fuse(bm25_hits, sem_hits)

res_hybrid = evaluate(hybrid_retrieve, 'Hybrid-RRF')
eval_df = pd.DataFrame([res_bm25, res_sem, res_hybrid])
print(eval_df)

       Method  MAP@10  MRR@10  nDCG@10
0        BM25  0.9292     1.0   0.9748
1    Semantic  1.0000     1.0   1.0000
2  Hybrid-RRF  0.9775     1.0   0.9912


### Milestone 5: LLM Generation (RAG)

In [15]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

def rag_answer(query):
    hits = hybrid_retrieve(query)

    context = ""
    for i, h in enumerate(hits):
        context += f"[Source {i+1}] {h['title']}\nExcerpt: {h['passage'][:200]}\n\n"

    prompt = (
        "You are a technical assistant.\n"
        "Answer using ONLY the sources provided. Cite as [Source N].\n\n"
        f"Question: {query}\n\nSources:\n{context}\n\nAnswer:"
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300,
        temperature=0.2
    )

    return response.choices[0].message.content, hits

test_q = 'What is the difference between AI and machine learning?'
ans, hits = rag_answer(test_q)
print(ans)

Artificial intelligence (AI) is the broad field that encompasses any technique enabling computers to mimic human intelligence—such as reasoning, planning, perception, and language understanding. Machine learning (ML) is a sub‑area of AI that focuses specifically on algorithms that allow systems to learn patterns from data and improve their performance automatically, without being explicitly programmed for each task. In other words, all machine‑learning methods are AI, but AI also includes many other approaches (e.g., symbolic reasoning, expert systems) that do not involve learning from data. [Source 2]


### Milestone 6: Gradio UI

In [16]:
import gradio as gr

def format_sources_html(hits):
    cards = []
    for h in hits:
        cards.append(
            "<div style='border:1px solid #ddd; padding:10px; margin:5px 0; border-radius:5px;'>"
            f"<b>[Source {h['rank']}]</b> <span style='color:#2c3e50'>{h['title']}</span><br>"
            f"<small>Tags: {h['tags']}</small><br>"
            f"<span style='font-size:0.9em'>{h['passage'][:200]}...</span>"
            "</div>"
        )
    return ''.join(cards)

def qa_handler(query):
    if not query:
        return "Please enter a question.", ""

    answer, hits = rag_answer(query)
    sources_html = format_sources_html(hits)
    return answer, sources_html

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔍 Hybrid RAG Q&A Assistant")
    gr.Markdown("BM25 + MiniLM + RRF Fusion | StackLite-6K Corpus")

    with gr.Tabs():
        with gr.Tab("Ask Question"):
            with gr.Row():
                with gr.Column(scale=2):
                    query_box = gr.Textbox(label="Your Question", lines=2)
                    ask_btn = gr.Button("Ask", variant="primary")
                with gr.Column(scale=3):
                    answer_box = gr.Textbox(label="Generated Answer", lines=6, interactive=False)

            sources_box = gr.HTML(label="Retrieved Sources")

            ask_btn.click(fn=qa_handler, inputs=[query_box], outputs=[answer_box, sources_box])

            gr.Examples([
                "What is self-supervised learning in machine learning?",
                "Why do transformers outperform RNNs in long-range dependencies?",
                "What is the difference between model-free and model-based RL?"
            ], inputs=[query_box])

        with gr.Tab("Evaluation Metrics"):
            gr.DataFrame(value=eval_df)

demo.launch(share=True)

/tmp/ipykernel_5139/1075672667.py:23: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://960c1953921c9566a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
